# 00b · Compatibility check (Step 0.3)

**Runtime: L4 GPU** (Runtime → Change runtime type → L4). Takes roughly 30–45 minutes.

Checks that everything the project depends on actually works on Colab's L4:
1. Builds an isolated Python environment with pinned versions (vLLM 0.32.0, transformers 5.19.0, torch 2.13 pulled in by vLLM). Colab's own preinstalled torch is left alone.
2. Downloads **Qwen3-VL-4B-Instruct** (~9 GB) and asks it one question about a test image drawn in code (a fake milk label, no VizWiz photos).
3. Creates a *blank* LoRA adapter (changes nothing, by design).
4. Starts a **vLLM** server with that adapter and checks base model and adapter give identical answers, which proves vLLM can serve LoRA adapters for this model.

Run cells **one at a time, top to bottom**, and paste back what each **Paste back** note asks for.

## 1 · Settings, Drive, code

In [ ]:
from google.colab import drive

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
VENV = "/content/venv"  # isolated, pinned GPU environment
PY = f"{VENV}/bin/python"
HF_LOCAL = "/content/hf_cache"  # local disk for this test; Drive vs local is decided from timings
LOGS = f"{DRIVE_ROOT}/logs"

drive.mount("/content/drive")
!if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi
!git -C {REPO_DIR} log --oneline -1
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

**Paste back:** the commit line and the GPU line. The GPU must say **NVIDIA L4**.

## 2 · Build the pinned environment
Uses `uv` (a fast pip replacement). Expect a few minutes; vLLM pulls in torch 2.13.

In [ ]:
%%time
!pip install -q uv
!uv venv -q --clear --python 3.13 {VENV}
!uv pip install -q --python {PY} -e {REPO_DIR} -r {REPO_DIR}/requirements/colab-gpu.txt
!uv pip freeze --python {PY} > {LOGS}/freeze_l4.txt
!{PY} -m vizwiz.compat versions

**Paste back:** the versions JSON and the `Wall time` line. Also upload `MyDrive/vizwiz/logs/freeze_l4.txt`. If the install fails, paste the full error.

## 3 · transformers check + blank LoRA adapter
Downloads the model (~9 GB) to local disk, loads it in bf16, answers one question, then saves a blank adapter.

In [ ]:
%%time
!HF_HOME={HF_LOCAL} {PY} -m vizwiz.compat hf --lora-out /content/blank_lora --report {LOGS}/compat_hf.json

**Paste back:** the JSON report (also saved as `MyDrive/vizwiz/logs/compat_hf.json`) and the `Wall time` line, or the full error.

## 4 · vLLM check with LoRA
Starts a vLLM server in the background (first start compiles kernels; can take several minutes), sends the same question to the base model and the blank adapter, then shuts the server down. Server output goes to `MyDrive/vizwiz/logs/vllm_server.log`.

In [ ]:
%%time
!HF_HOME={HF_LOCAL} {PY} -m vizwiz.compat vllm --lora /content/blank_lora --report {LOGS}/compat_vllm.json --log {LOGS}/vllm_server.log

**Paste back:** the JSON report (`MyDrive/vizwiz/logs/compat_vllm.json`) and the `Wall time` line. If `ok` is `false`, also upload `vllm_server.log`.

## Done
Upload or paste back from `MyDrive/vizwiz/logs/`:
- `freeze_l4.txt`
- `compat_hf.json`
- `compat_vllm.json`
- `vllm_server.log` (only if something failed)

Then you can disconnect the runtime (Runtime → Disconnect and delete runtime) to stop using compute units.